<a href="https://colab.research.google.com/github/Harshini280905/Electricity-demand-forecast/blob/main/first_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Electricity demand forecast


## LOAD DATA

In [ ]:
import pandas as pd
df = pd.read_csv('elect demand.csv')
print(df.columns)
print(df.head())



##DATA PREPARATION
######DATA SEPERATION AS X AND Y

In [ ]:

df['date_1'] = pd.to_datetime(df['date_1'], format='%d/%m/%Y', errors='coerce')
df['date_2'] = pd.to_datetime(df['date_2'], format='%d/%m/%Y', errors='coerce')
df['date_1_day'] = df['date_1'].dt.day
df['date_1_month'] = df['date_1'].dt.month
df['date_1_year'] = df['date_1'].dt.year

df['date_2_day'] = df['date_2'].dt.day
df['date_2_month'] = df['date_2'].dt.month
df['date_2_year'] = df['date_2'].dt.year
df.drop(['date_1', 'date_2'], axis=1, inplace=True)
# Step 1: Drop rows where either x or y is NaN
df_cleaned = df.dropna()

# Step 2: Separate features and label
x = df_cleaned.drop('total_demand(mw)', axis=1)
y = df_cleaned['total_demand(mw)']



####datasplit

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=100)


##BUILD MODEL


###LINEAR REGRESSION

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(x_train, y_train)


#####Applying model to make prediction

In [ ]:
y_lr_train_pred = lr.predict(x_train)
y_lr_test_pred=lr.predict(x_test)

In [ ]:
y_lr_train_pred

In [ ]:
y_lr_test_pred

#####Evaluate model performance

In [ ]:
from sklearn.metrics import mean_squared_error,r2_score
lr_train_mse=mean_squared_error(y_train,y_lr_train_pred)
lr_train_r2=r2_score(y_train,y_lr_train_pred)
lr_test_mse=mean_squared_error(y_test,y_lr_test_pred)
lr_test_r2=r2_score(y_test,y_lr_test_pred)

In [ ]:
print('LR MSE TRAIN:',lr_train_mse)
print('LR r2 TRAIN:',lr_train_r2)
print('LR MSE Test:',lr_test_mse)
print('LR r2 Test:',lr_test_r2)


In [ ]:
lr_results=pd.DataFrame(['Linear regression',lr_train_mse,lr_train_r2,lr_test_mse,lr_test_r2]).transpose()
lr_results.columns=['Method','Training MSE','Training R2','Testing MSE','Testing R2']

In [ ]:
lr_results

####VISUALIZE

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Actual vs Predicted Plot (TEST SET)
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=y_lr_test_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Total Demand (MW)")
plt.ylabel("Predicted Total Demand (MW)")
plt.title("Actual vs Predicted - Test Set")
plt.grid(True)
plt.show()

# 2. Residual Plot (TEST SET)
residuals = y_test - y_lr_test_pred
plt.figure(figsize=(8,6))
sns.histplot(residuals, kde=True, bins=30)
plt.xlabel("Prediction Error (Residuals)")
plt.title("Distribution of Residuals - Test Set")
plt.grid(True)
plt.show()

# 3. Feature Coefficients (Impact of each feature)
coeff_df = pd.DataFrame({
    'Feature': x.columns,
    'Coefficient': lr.coef_
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=coeff_df, x='Coefficient', y='Feature')
plt.title("Feature Importance (Linear Regression Coefficients)")
plt.grid(True)
plt.show()


###RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestRegressor
'''not classifier coz y is a quantitative value and not a categorcal value'''
rf=RandomForestRegressor(max_depth=2,random_state=100)
rf.fit(x_train,y_train)

#####Applying model to make prediction

In [ ]:
y_rf_train_pred = rf.predict(x_train)
y_rf_test_pred=rf.predict(x_test)

In [ ]:
from sklearn.metrics import mean_squared_error,r2_score
rf_train_mse=mean_squared_error(y_train,y_rf_train_pred)
rf_train_r2=r2_score(y_train,y_rf_train_pred)
rf_test_mse=mean_squared_error(y_test,y_rf_test_pred)
rf_test_r2=r2_score(y_test,y_rf_test_pred)

In [ ]:
rf_results=pd.DataFrame(['Linear regression',rf_train_mse,rf_train_r2,rf_test_mse,rf_test_r2]).transpose()
rf_results.columns=['Method','Training MSE','Training R2','Testing MSE','Testing R2']
rf_results

###XGBOOST

In [ ]:
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.08,
                    subsample=0.9, colsample_bytree=0.9, random_state=100)
xgb.fit(x_train, y_train)


#####Applying model to make prediction

In [ ]:
y_xgb_train_pred = xgb.predict(x_train)
y_xgb_test_pred = xgb.predict(x_test)


#####Evaluate model performance

In [ ]:
xgb_train_mse = mean_squared_error(y_train, y_xgb_train_pred)
xgb_train_r2 = r2_score(y_train, y_xgb_train_pred)
xgb_test_mse = mean_squared_error(y_test, y_xgb_test_pred)
xgb_test_r2 = r2_score(y_test, y_xgb_test_pred)


In [ ]:
print('XGB MSE TRAIN:', xgb_train_mse)
print('XGB r2 TRAIN:', xgb_train_r2)
print('XGB MSE Test:', xgb_test_mse)
print('XGB r2 Test:', xgb_test_r2)


In [ ]:
xgb_results=pd.DataFrame(['XGBoost',xgb_train_mse,xgb_train_r2,xgb_test_mse,xgb_test_r2]).transpose()
xgb_results.columns=['Method','Training MSE','Training R2','Testing MSE','Testing R2']
xgb_results


#####Feature importance

In [ ]:
import matplotlib.pyplot as plt
importances = pd.Series(xgb.feature_importances_, index=x_train.columns).sort_values(ascending=False)
plt.figure(figsize=(8,6))
importances.head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance (Top 10)')
plt.tight_layout()
plt.show()


##MODEL COMPARISON

In [ ]:
df_models=pd.concat([lr_results,rf_results,xgb_results],axis=0)
'''axis=0 means row wise else col wise for 1'''
df_models


In [ ]:
df_models.reset_index(drop=True)

**Insight:** XGBoost outperforms both Linear Regression and Random Forest on this dataset (higher Test R2, lower Test MSE), capturing non-linear interactions between weather and time features that Linear Regression misses, without underfitting the way the depth-limited Random Forest does.

#DATA VISUALIZATION OF PREDICTION RESULTS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.scatter(x=y_train,y=y_lr_train_pred,c="#7CAE00",alpha=0.3)
z=np.polyfit(y_train,y_lr_train_pred,1)
p=np.poly1d(z)
plt.plot(y_train,p(y_train),'#F8766D')
plt.ylabel('Predict TOTAL DEMAND')
plt.xlabel('Actual TOTAL DEMAND')